Загружаем готовый датасет с отзывами на Яндекс Картах за 2023 год

In [13]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
df = pd.read_csv('geo-reviews-dataset-2023.csv')


Saving geo-reviews-dataset-2023.csv to geo-reviews-dataset-2023 (1).csv


In [22]:
df.head()
df.info()
df.describe(include='all')
print(f"Строк: {df.shape[0]}, Колонок: {df.shape[1]}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   address  500000 non-null  object 
 1   name_ru  499030 non-null  object 
 2   rating   500000 non-null  float64
 3   rubrics  500000 non-null  object 
 4   text     500000 non-null  object 
dtypes: float64(1), object(4)
memory usage: 19.1+ MB
Строк: 500000, Колонок: 5


Смотрим какие рубрики встречаются чаще всего и выбираем топ три для дальнейшей работы

In [23]:
print(df['rubrics'].head(10))
import ast
from collections import Counter

def extract_rubrics(rubric_str):
    try:

        return ast.literal_eval(rubric_str)
    except:

        return [x.strip() for x in str(rubric_str).split(',')]

all_rubrics = []
for rub in df['rubrics'].dropna():
    all_rubrics.extend(extract_rubrics(rub))

rubric_counts = Counter(all_rubrics)

print("\nТоп-30 рубрик:")
for rubric, count in rubric_counts.most_common(30):
    print(f"  {rubric}: {count}")
print(f"\nВсего уникальных рубрик: {len(rubric_counts)}")


0                                       Жилой комплекс
1    Магазин продуктов;Продукты глубокой заморозки;...
2                                          Фитнес-клуб
3          Пункт проката;Прокат велосипедов;Сапсёрфинг
4    Салон красоты;Визажисты, стилисты;Салон бровей...
5            Оператор сотовой связи;Интернет-провайдер
6                                                 Кафе
7    Вейп-шоп;Магазин табака и курительных принадле...
8                                         Кафе;Кофейня
9                                         Кафе;Кофейня
Name: rubrics, dtype: object

Топ-30 рубрик:
  Гостиница: 42242
  Ресторан: 14615
  Кафе: 12366
  Супермаркет: 8899
  Бар: 7528
  Ресторан;Бар: 6280
  Автосервис: 6206
  Медцентр: 5904
  паб: 5572
  Магазин продуктов: 5289
  Музей: 5005
  Быстрое питание: 4902
  Ресторан;Кафе: 4554
  Супермаркет;Магазин продуктов: 4541
  Пункт выдачи: 4316
  Аптека: 4114
  автотехцентр: 3805
  Парк культуры и отдыха: 3777
  Достопримечательность: 3760
  База: 375

Удаляем адрес, чтобы облегчить датасет

In [27]:
print(df.columns.tolist())

['address', 'name_ru', 'rating', 'rubrics', 'text']


In [28]:
df = df.drop(columns=['address'])
print(df.columns.tolist())

['name_ru', 'rating', 'rubrics', 'text']


In [31]:
selected_categories = ['Гостиница', 'Ресторан', 'Кафе']
df_filtered = df[df['rubrics'].fillna('').str.lower().str.contains('|'.join(selected_categories).lower())].copy()
print(f"Отфильтровано: {len(df_filtered)} строк")

Отфильтровано: 136421 строк


Создаём золотой стандарт из 100 сгенерированных рекламных  отзывов и 500 реальных отзывов с датасета и загружаем его

In [32]:
import pandas as pd
import random

# Шаблоны рекламных фейков
fake_templates = [
    "Очень крутое место! Всем рекомендую! {name} просто топ!",
    "Лучший {category} в городе! Обязательно посетите! Акции каждый день!",
    "Отличный сервис! Приходите в {name} — не пожалеете! Скидки постоянным клиентам!",
    "Восторг! {name} — это любовь с первого визита. Самое лучшее место!",
    "Потрясающе! Обязательно к посещению! {name} задаёт тренды в {city}",
    "Невероятная атмосфера! Советую всем друзьям! {name} ждёт вас!",
    "Топ заведение! Лучшее соотношение цена-качество. Всем советую {name}!",
    "Классное место! Обязательно приду ещё раз. {name} — огонь!"
]

real_names = df_filtered['name_ru'].dropna().unique().tolist()
real_categories = df_filtered['rubrics'].dropna().unique().tolist()

# Генерируем 100 фейковых отзывов
fake_reviews = []
for i in range(100):
    template = random.choice(fake_templates)
    name = random.choice(real_names)
    category = random.choice(real_categories)

    fake_text = template.format(name=name, category=category, city='Москва')

    fake_reviews.append({
        'text': fake_text,
        'rating': random.choice([4, 5]),
        'name_ru': name,
        'rubrics': category,
        'is_fake': 1  # метка, что это реклама
    })

df_fakes = pd.DataFrame(fake_reviews)

# Добавляем реальные отзывы
df_real = df_filtered.sample(n=500, random_state=42).copy()
df_real['is_fake'] = 0
df_mixed = pd.concat([df_real, df_fakes], ignore_index=True)

df_mixed = df_mixed.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Реальных отзывов: {len(df_real)}")
print(f"Фейковых (реклама): {len(df_fakes)}")
print(f"Всего: {len(df_mixed)}")

df_mixed.to_excel('Датасет_с_рекламой.xlsx', index=False)

def count_ad_markers(text):
    markers = {
        'призывы': ['рекомендую', 'советую', 'обязательно', 'приходите', 'посетите'],
        'восторги': ['лучший', 'топ', 'огонь', 'класс', 'супер', 'отлично', 'восторг'],
        'отсутствие деталей': len(text.split()) < 10,
        'высокая оценка': True
    }

    score = 0
    for marker_type, keywords in [('призывы', markers['призывы']), ('восторги', markers['восторги'])]:
        for kw in keywords:
            if kw in text.lower():
                score += 1

    if markers['отсутствие деталей']:
        score += 1

    return score

df_filtered['ad_score'] = df_filtered['text'].astype(str).apply(count_ad_markers)
print(f"Отзывы с высоким рекламным скором (>3): {len(df_filtered[df_filtered['ad_score'] > 3])}")

Реальных отзывов: 500
Фейковых (реклама): 100
Всего: 600
Отзывы с высоким рекламным скором (>3): 316


In [33]:
from google.colab import files
files.download('Датасет_с_рекламой.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
import pandas as pd

df = pd.read_excel('Датасет_с_рекламой.xlsx')

print("Распределение меток:")
print(df['is_fake'].value_counts())
print(f"\nРеальных отзывов (0): {len(df[df['is_fake']==0])}")
print(f"Рекламных фейков (1): {len(df[df['is_fake']==1])}")
print(f"Доля фейков: {len(df[df['is_fake']==1])/len(df)*100:.1f}%")
print()
# Показываем 10 примеров рекламных отзывов
print("ПРИМЕРЫ РЕКЛАМНЫХ ФЕЙКОВ (is_fake=1):")
print()

fakes = df[df['is_fake']==1]
for i, row in fakes.head(10).iterrows():
    print(f"Заведение: {row['name_ru']}")
    print(f"Текст: {row['text'][:150]}...")
    print("-"*40)


print()
print("ПРИМЕРЫ РЕАЛЬНЫХ ОТЗЫВОВ (is_fake=0):")
print()

reals = df[df['is_fake']==0]
for i, row in reals.head(5).iterrows():
    print(f"Оценка: {row['rating']}")
    print(f"Текст: {row['text'][:200]}...")
    print("-"*40)

Распределение меток:
is_fake
0    500
1    100
Name: count, dtype: int64

Реальных отзывов (0): 500
Рекламных фейков (1): 100
Доля фейков: 16.7%

ПРИМЕРЫ РЕКЛАМНЫХ ФЕЙКОВ (is_fake=1):

Заведение: Shalet Бойка
Текст: Классное место! Обязательно приду ещё раз. Shalet Бойка — огонь!...
----------------------------------------
Заведение: Вояж
Текст: Классное место! Обязательно приду ещё раз. Вояж — огонь!...
----------------------------------------
Заведение: Фарш
Текст: Невероятная атмосфера! Советую всем друзьям! Фарш ждёт вас!...
----------------------------------------
Заведение: Бастон
Текст: Очень крутое место! Всем рекомендую! Бастон просто топ!...
----------------------------------------
Заведение: Asiainn
Текст: Невероятная атмосфера! Советую всем друзьям! Asiainn ждёт вас!...
----------------------------------------
Заведение: Ерзовка
Текст: Восторг! Ерзовка — это любовь с первого визита. Самое лучшее место!...
----------------------------------------
Заведение: Парк-отель Торбее